# XGBoost age bands on later transactions

Train exact-age, age-band, and no-age XGBoost models on earlier transactions and evaluate them on later transactions. Keep the features, age bands, model settings, age audit groups, and 0.6% review budget from [the random-split experiment](07_age_band_xgboost.ipynb). Compare the overall detection trade-off and the change in recall among ages 25-39.

Run with pandas, NumPy, scikit-learn, and XGBoost installed and the project's `data/fraudTrain.csv` in place.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

path = Path("data/fraudTrain.csv")
if not path.is_file():
    path = Path("../data/fraudTrain.csv")

transactions = pd.read_csv(
    path,
    usecols=[
        "trans_date_trans_time", "dob", "amt", "category", "lat", "long",
        "merch_lat", "merch_long", "is_fraud",
    ],
    parse_dates=["trans_date_trans_time", "dob"],
)
timestamp = transactions["trans_date_trans_time"]
transactions["hour"] = timestamp.dt.hour
transactions["day_of_week"] = timestamp.dt.day_name()
# Match the age calculation used by the existing XGBoost baseline.
transactions["age"] = ((timestamp - transactions["dob"]).dt.days / 365.25).astype(int)

lat, lon, merchant_lat, merchant_lon = (
    np.radians(transactions[column])
    for column in ("lat", "long", "merch_lat", "merch_long")
)
a = (
    np.sin((merchant_lat - lat) / 2) ** 2
    + np.cos(lat) * np.cos(merchant_lat) * np.sin((merchant_lon - lon) / 2) ** 2
)
transactions["distance_km"] = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

age_bins = [0, 20, 30, 40, 50, 60, 70, 80, np.inf]
age_labels = ["Under 20", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80+"]
transactions["age_band"] = pd.cut(
    transactions["age"], age_bins, labels=age_labels, right=False,
)
if transactions["age_band"].isna().any():
    raise ValueError("Every transaction must fit one of the age bands.")

numeric = ["amt", "hour", "distance_km"]
categorical = ["category", "day_of_week"]
features = transactions[numeric + categorical + ["age", "age_band"]]
ordered_timestamps = timestamp.sort_values(kind="stable")
cutoff = ordered_timestamps.iloc[int(0.8 * len(transactions))]
# Keep all transactions at the cutoff timestamp together in the holdout.
is_earlier = timestamp < cutoff
train, test = features.loc[is_earlier], features.loc[~is_earlier]
y_train = transactions.loc[train.index, "is_fraud"]
y_test = transactions.loc[test.index, "is_fraud"]
assert timestamp.loc[train.index].max() < timestamp.loc[test.index].min()
assert len(train) + len(test) == len(transactions)

print(f"Training period: {timestamp.loc[train.index].min()} to {timestamp.loc[train.index].max()}")
print(f"Holdout period: {cutoff} to {timestamp.loc[test.index].max()}")
print(f"Training transactions: {len(train):,}")
print(f"Held-out transactions: {len(test):,}, including {y_test.sum():,} fraud cases")

Training period: 2019-01-01 00:00:18 to 2020-03-06 07:15:17
Holdout period: 2020-03-06 07:16:43 to 2020-06-21 12:13:37
Training transactions: 1,037,340
Held-out transactions: 259,335, including 1,538 fraud cases


## Matched alert volume

Each model uses the settings from the final XGBoost notebook and trains on the earliest approximately 80% of transactions. The simulated review budget is 0.6% of held-out transactions. Each model alerts on its highest scores up to that budget. The comparison represents batch ranking under a fixed review capacity. Rows with tied scores are ordered by their fixed position in the held-out split.

In [2]:
variants = {
    "Exact age": (["amt", "hour", "age", "distance_km"], categorical),
    "Age band": (numeric, categorical + ["age_band"]),
    "No age": (numeric, categorical),
}
alert_fraction = 0.006
alert_count = round(alert_fraction * len(test))
actual_fraud = y_test.to_numpy() == 1
predictions = {}
overall_rows = []

for name, (numeric_columns, categorical_columns) in variants.items():
    columns = numeric_columns + categorical_columns
    preprocessor = ColumnTransformer([
        ("numeric", StandardScaler(), numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ])
    model = XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05,
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        random_state=42, n_jobs=4,
    )
    scorer = make_pipeline(preprocessor, model)
    scorer.fit(train[columns], y_train)
    scores = scorer.predict_proba(test[columns])[:, 1]

    if name == "Age band":
        # Exercise the scoring path with a supplied band and no birth date or exact age.
        example = test[columns].iloc[[0]].copy()
        example["age_band"] = example["age_band"].astype(str)
        np.testing.assert_allclose(scorer.predict_proba(example)[0, 1], scores[0])
        age_band_scorer = scorer
        age_band_inputs = columns

    order = np.argsort(-scores, kind="stable")
    alerted = np.zeros(len(test), dtype=bool)
    alerted[order[:alert_count]] = True
    assert alerted.sum() == alert_count
    predictions[name] = alerted

    caught = np.count_nonzero(alerted & actual_fraud)
    overall_rows.append({
        "Model": name,
        "Alerts": alert_count,
        "Frauds caught": caught,
        "False alerts": alert_count - caught,
        "Precision (%)": 100 * caught / alert_count,
        "Recall (%)": 100 * caught / actual_fraud.sum(),
        "PR-AUC (average precision)": average_precision_score(actual_fraud, scores),
    })
    print(f"Evaluated {name}", flush=True)

overall = pd.DataFrame(overall_rows).set_index("Model")
display(overall.round({"Precision (%)": 2, "Recall (%)": 2, "PR-AUC (average precision)": 4}))

Evaluated Exact age
Evaluated Age band
Evaluated No age


,Alerts,Frauds caught,False alerts,Precision (%),Recall (%),PR-AUC (average precision)
Model,,,,,,
Exact age,1556,1286,270,82.65,83.62,0.9051
Age band,1556,1274,282,81.88,82.83,0.8950
No age,1556,1180,376,75.84,76.72,0.8420


## Age-group outcomes

Use the four broader audit groups from the age-removal notebook for every model. Exact age is used here to define common audit groups, outside the scoring path. False-positive rate is the share of legitimate transactions flagged. Recall is the share of fraud transactions flagged. The gap table measures the difference between the highest and lowest group rates in percentage points. Inspect group rates and denominators alongside the gaps, since a gap can shrink when outcomes worsen for a previously better-performing group.

In [3]:
audit_bins = [0, 25, 40, 60, np.inf]
audit_labels = ["Under 25", "25-39", "40-59", "60+"]
audit_groups = pd.cut(
    test["age"], audit_bins, labels=audit_labels, right=False,
).to_numpy()
group_rows = []

for group in audit_labels:
    legitimate = (audit_groups == group) & ~actual_fraud
    fraudulent = (audit_groups == group) & actual_fraud
    assert legitimate.any() and fraudulent.any()
    row = {
        "Age group": group,
        "Legitimate": legitimate.sum(),
        "Fraud": fraudulent.sum(),
    }
    for name, alerted in predictions.items():
        row[f"{name} FPR (%)"] = 100 * alerted[legitimate].mean()
        row[f"{name} recall (%)"] = 100 * alerted[fraudulent].mean()
    group_rows.append(row)

groups = pd.DataFrame(group_rows).set_index("Age group")
assert groups["Fraud"].sum() == actual_fraud.sum()
assert groups["Legitimate"].sum() == (~actual_fraud).sum()
display(groups.round(3))

gaps = pd.DataFrame([
    {
        "Model": name,
        "FPR gap (pp)": groups[f"{name} FPR (%)"].max() - groups[f"{name} FPR (%)"].min(),
        "Recall gap (pp)": groups[f"{name} recall (%)"].max() - groups[f"{name} recall (%)"].min(),
    }
    for name in variants
]).set_index("Model")
display(gaps.round(3))

,Legitimate,Fraud,Exact age FPR (%),Exact age recall (%),Age band FPR (%),Age band recall (%),No age FPR (%),No age recall (%)
Age group,,,,,,,,
Under 25,23699,87,0.135,81.609,0.173,79.310,0.236,78.161
25-39,81669,468,0.159,73.718,0.137,72.863,0.152,68.803
40-59,95922,541,0.095,85.028,0.105,84.288,0.138,78.928
60+,56507,442,0.030,92.760,0.050,92.308,0.113,82.127


,FPR gap (pp),Recall gap (pp)
Model,,
Exact age,0.129,19.042
Age band,0.123,19.444
No age,0.123,13.323


## Did the random-split findings repeat?

The earlier random split showed a 2.33 percentage point recall loss overall and a 4.94 point loss for ages 25-39 when switching from exact age to bands. Compare those differences with the later-period results below. The percentages describe this simulated dataset at one review budget. Cards can appear in both periods.

In [4]:
changes = pd.DataFrame({
    "Random holdout (pp)": [
        100 * (1245 - 1280) / 1501,
        100 * (292 - 312) / 405,
    ],
    "Later holdout (pp)": [
        overall.loc["Age band", "Recall (%)"] - overall.loc["Exact age", "Recall (%)"],
        groups.loc["25-39", "Age band recall (%)"] - groups.loc["25-39", "Exact age recall (%)"],
    ],
}, index=["Overall recall", "Recall for ages 25-39"])
display(changes.round(2))

,Random holdout (pp),Later holdout (pp)
Overall recall,-2.33,-0.78
Recall for ages 25-39,-4.94,-0.85


## Interpretation

At the same 1,556 alerts, the age-band model caught 12 fewer fraud cases than the exact-age model and 94 more than the no-age model. The recall loss for ages 25-39 repeated at a smaller magnitude, from 4.94 percentage points on the random holdout to 0.85 on the later holdout. On later transactions, ages 25-39 still had the lowest recall for all three models. The age-band model slightly narrowed the false-positive rate gap and slightly widened the recall gap compared with exact age. These results support age bands as a prototype input when reducing age detail is the priority. A broader fairness claim would require more evidence.

## Save and reuse the evaluated age-band model

Run the export cell after training to save the fitted preprocessor and XGBoost together. In a trusted local environment, load the bundle with `joblib.load` and call `bundle["pipeline"].predict_proba(inputs)[:, 1]` on a DataFrame with the columns in `bundle["features"]`. Use one of `bundle["age_bands"]` for `age_band`. The evaluation flagged the top 0.6% of scores within a batch. The saved pipeline returns fraud scores and leaves alert selection to the caller.

In [5]:
import joblib

model_path = path.parent.parent / "models" / "xgboost_age_band.pkl"
model_path.parent.mkdir(exist_ok=True)
bundle = {
    "pipeline": age_band_scorer,
    "features": tuple(age_band_inputs),
    "age_bands": tuple(age_labels),
    "training_cutoff": cutoff.isoformat(),
    "review_fraction": alert_fraction,
}
joblib.dump(bundle, model_path)

# Reload and verify that scores match with only the published input columns.
saved = joblib.load(model_path)
sample = test.loc[:, list(saved["features"])].head(100).copy()
sample["age_band"] = sample["age_band"].astype(str)
np.testing.assert_array_equal(
    saved["pipeline"].predict_proba(sample),
    age_band_scorer.predict_proba(sample),
)
print(f"Saved and verified: {model_path.name}")

Saved and verified: xgboost_age_band.pkl
